In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/hc3_all.csv", header=0)

if "id" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "id"})

if "id" not in df.columns:
    raise ValueError("The raw HC3 dataset must contain an id or index column.")

if df["id"].isna().any() or df["id"].duplicated().any():
    df["id"] = range(len(df))

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (24322, 5)


,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,reddit_eli5
4,4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,reddit_eli5


## Find missing values

In [3]:
# Are there missing values?
print(df.isnull().values.any())
print(df.isnull().head())

nan_count = np.sum(df.isnull(), axis = 0)
print(f"Dataset NaN count:\n{nan_count}")


missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing_summary

False
      id  question  human_answers  chatgpt_answers  source
0  False     False          False            False   False
1  False     False          False            False   False
2  False     False          False            False   False
3  False     False          False            False   False
4  False     False          False            False   False
Dataset NaN count:
id                 0
question           0
human_answers      0
chatgpt_answers    0
source             0
dtype: int64


,missing_count,missing_percent
id,0,0.0
question,0,0.0
human_answers,0,0.0
chatgpt_answers,0,0.0
source,0,0.0


We actually shouldn't need to correct any missing data in the source column as the data in that column are strings, and it doesn't include particularly important information for this project.

## Find duplicates

In [4]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

Duplicate rows: 0


,id,question,human_answers,chatgpt_answers,source


## Find inconsistencies

In [5]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    series = df[col].dropna().astype(str)

    whitespace_issues = series[
        series.str.strip() != series
    ]

    multiple_space_issues = series[
        series.str.contains(r"\s{2,}", regex=True)
    ]

    print(f"\n{col} Column:")
    print(f"Leading/trailing whitespace: {len(whitespace_issues)}")
    print(f"Multiple spaces: {len(multiple_space_issues)}")


question Column:
Leading/trailing whitespace: 1944
Multiple spaces: 213

human_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 19936

chatgpt_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 1843

source Column:
Leading/trailing whitespace: 0
Multiple spaces: 0


### Check for empty lists

In [6]:
print("Empty human answer lists:")
print((df["human_answers"] == "[]").sum())

print("\nEmpty ChatGPT answer lists:")
print((df["chatgpt_answers"] == "[]").sum())

Empty human answer lists:
0

Empty ChatGPT answer lists:
455


### Remove rows without a ChatGPT answer

In [7]:
df = df[df["chatgpt_answers"] != "[]"].copy()

print("Rows after removing empty ChatGPT answers:", len(df))

Rows after removing empty ChatGPT answers: 23867


### Verify no empty lists

In [8]:
print("Empty human answer lists:",
      (df["human_answers"] == "[]").sum())

print("Empty ChatGPT answer lists:",
      (df["chatgpt_answers"] == "[]").sum())

Empty human answer lists: 0
Empty ChatGPT answer lists: 0


## Create the cleaned dataset

In [10]:
clean = df.copy()


# Clean text
text_columns = clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    clean[col] = (
        clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

## Create the expanded answer-level dataset

The cleaned dataset remains one row per question for leakage-safe splitting.

This separate dataset expands the human and ChatGPT answer lists into one labeled row per answer.

In [11]:
import ast
import re
import unicodedata

def parse_answer_list(value):
    if isinstance(value, list):
        return value

    repaired = re.sub(r"(?<=[\x27\x22])\s+(?=[\x27\x22])", ", ", value)
    return ast.literal_eval(repaired)

def normalize_text(text):
    if not isinstance(text, str):
        return text

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Standardize curly quotes/apostrophes
    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Fix spaces immediately around apostrophes
    # don ' t -> don't
    text = re.sub(r"\s*'\s*", "'", text)

    # Fix tokenized contractions
    # do n't -> don't
    # I 'm -> I'm
    # you 're -> you're
    text = re.sub(
        r"\b(\w+)\s+(n't|'s|'m|'re|'ve|'ll|'d)\b",
        r"\1\2",
        text,
        flags=re.IGNORECASE
    )

    # Remove spaces before punctuation
    # word . -> word.
    text = re.sub(r"\s+([.,!?;:%])", r"\1", text)

    return text.strip()

parts = []

for column, label in [("human_answers", "human"), ("chatgpt_answers", "chatgpt")]:
    part = clean[["id", "question", "source", column]].copy()
    part[column] = part[column].apply(parse_answer_list)
    part = part.explode(column).rename(columns={column: "text"})
    part["label"] = label
    part["target"] = part["label"].map({"human": 0, "chatgpt": 1})
    parts.append(part)

expanded = pd.concat(parts, ignore_index=True)

expanded = expanded[expanded["text"].notna()].copy()

In [12]:
# Temporary audit copy
expanded["original_text"] = expanded["text"]

# Normalize answer text
expanded["text"] = expanded["text"].apply(normalize_text)

# Remove empty answers
expanded = (expanded[expanded["text"].str.strip().ne("")].reset_index(drop=True))

# Inspect what changed
changed = expanded[expanded["original_text"] != expanded["text"]]

print(
    f"Normalized {len(changed):,} "
    f"of {len(expanded):,} answers."
)

display(changed[["source","label","original_text","text"]].head(50))

# Remove audit column before saving
expanded = expanded.drop(columns=["original_text"])

Normalized 60,381 of 84,066 answers.


,source,label,original_text,text
0,reddit_eli5,human,"Basically there are many categories of "" Best ...","Basically there are many categories of "" Best ..."
1,reddit_eli5,human,"If you 're hearing about it , it 's because it...","If you're hearing about it, it's because it wa..."
2,reddit_eli5,human,"One reason is lots of catagories . However , h...","One reason is lots of catagories. However, how..."
3,reddit_eli5,human,salt is good for not dying in car crashes and ...,salt is good for not dying in car crashes and ...
4,reddit_eli5,human,"In Minnesota and North Dakota , they tend to u...","In Minnesota and North Dakota, they tend to us..."
5,reddit_eli5,human,"Used to work in the salt industry ( yes , it '...","Used to work in the salt industry ( yes, it's ..."
6,reddit_eli5,human,The way it works is that old TV stations got a...,The way it works is that old TV stations got a...
7,reddit_eli5,human,HD does n't look like anything at all on an SD...,HD doesn't look like anything at all on an SD ...
8,reddit_eli5,human,There are a few reasons why SD channels still ...,There are a few reasons why SD channels still ...
9,reddit_eli5,human,You ca n't just go around assassinating the le...,You can't just go around assassinating the lea...


## Saving the dataset

In [14]:
expanded_path = "../../data/processed/expanded_dataset.csv"
expanded.to_csv(expanded_path, index=False, chunksize=1000)

print(expanded["label"].value_counts())

print(f"Saved cleaned dataset to: {expanded_path}")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {expanded.shape}")

label
human      57181
chatgpt    26885
Name: count, dtype: int64
Saved cleaned dataset to: ../../data/processed/expanded_dataset.csv
Original shape: (23867, 5)
Cleaned shape: (84066, 6)
